# 02 · DenseNet-161: supervised CNN baseline
**OMARI Seminar Series — From Pixels to Predictions: Developing AI Models for Medical Image Interpretation**

Load ImageNet-pretrained DenseNet-161, replace the 1000-way head with a 6-way head, run frozen-backbone and full fine-tuning experiments, and export feature maps for the website.

| | |
|---|---|
| **Runtime** | GPU recommended: demo ≈15 min (T4), full ≈1–2 h per regime |
| **Needs** | `splits/split_manifest.csv` from notebook 01 (rebuilt automatically if absent) |
| **Produces** | `runs/densenet161_*/{best.pt, history.json, test_predictions.npz, summary.json}` |
| **Modes** | `smoke` (synthetic data, random weights: checks the code only) · `demo` (real data subset, few epochs) · `full` (complete experiment) |

> Results from `smoke` mode are meaningless by construction and must never be exported to the website.
> Every artifact written by these notebooks records the script, checkpoint hash and parameters that produced it.

In [ ]:
#@title 0 · Setup: repository, packages, data and artifact folders { display-mode: "form" }
import os, sys, subprocess, json, random, time
MODE = "demo"  #@param ["smoke", "demo", "full"]
REPO_URL = "https://github.com/YOUR-ORG/omari-workshop.git"  #@param {type:"string"}
DATA_ROOT = "/content/data/FETAL_PLANES_ZENODO"  #@param {type:"string"}
ARTIFACTS = "/content/omari_artifacts"  #@param {type:"string"}
USE_DRIVE_FOR_ARTIFACTS = False  #@param {type:"boolean"}
MODE = os.environ.get("OMARI_MODE", MODE)            # lets CI / nbclient override
ARTIFACTS = os.environ.get("OMARI_ARTIFACTS", ARTIFACTS)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_DRIVE_FOR_ARTIFACTS:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/omari_artifacts"

# 1) make `omari_us` importable: installed package > local checkout (../python) > clone REPO_URL
def _find_repo():
    for cand in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/omari-workshop"]:
        if os.path.isdir(os.path.join(cand, "python", "omari_us")):
            return cand
    return None
REPO = _find_repo()
if REPO is None:
    if "YOUR-ORG" in REPO_URL:
        raise RuntimeError("Set REPO_URL to the repository that contains python/omari_us (or upload the repo to /content/omari-workshop).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/omari-workshop"], check=True)
    REPO = "/content/omari-workshop"
sys.path.insert(0, os.path.join(REPO, "python"))

# 2) packages (Colab already has torch/torchvision; timm and friends are added)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9.12", "scikit-learn", "scipy", "pyyaml", "opencv-python-headless", "gdown"], check=True)

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import omari_us
from omari_us.utils import seed_everything, get_device, CLASSES, CLASS_SHORT
SEED = 42
seed_everything(SEED)
DEVICE = get_device()
os.makedirs(ARTIFACTS, exist_ok=True)
SPLITS_CSV = os.path.join(ARTIFACTS, "splits", "split_manifest.csv")
RUNS = os.path.join(ARTIFACTS, "runs")
print(f"omari_us {omari_us.__version__} | torch {torch.__version__} | device {DEVICE} | MODE={MODE} | artifacts → {ARTIFACTS}")
if DEVICE.type == "cpu" and MODE != "smoke":
    print("⚠ No GPU. In Colab: Runtime → Change runtime type → T4 GPU. CPU training of these models is impractically slow.")

In [ ]:
#@title 1 · Data: download FETAL_PLANES_DB (2.1 GB, CC BY 4.0) or build the synthetic smoke dataset { display-mode: "form" }
from omari_us.datasets import read_metadata, make_splits, synthetic_metadata, save_split_manifest, load_split_manifest, ZENODO_URL, ZENODO_MD5

def ensure_fetal_planes(root=DATA_ROOT):
    """Download + verify + extract the Zenodo archive if the metadata CSV is not already present."""
    import glob, hashlib
    if glob.glob(os.path.join(root, "**", "FETAL_PLANES_DB_data.csv"), recursive=True):
        return
    os.makedirs(root, exist_ok=True)
    z = os.path.join(os.path.dirname(root), "FETAL_PLANES_ZENODO.zip")
    if not os.path.exists(z):
        print("downloading", ZENODO_URL)
        subprocess.run(["curl", "-L", "--fail", "-o", z, ZENODO_URL], check=True)
    md5 = hashlib.md5(open(z, "rb").read()).hexdigest()
    assert md5 == ZENODO_MD5, f"md5 mismatch: {md5}"
    subprocess.run(["unzip", "-q", "-n", z, "-d", root], check=True)

if MODE == "smoke":
    SMOKE_DIR = os.path.join(ARTIFACTS, "smoke_data")
    meta = synthetic_metadata(60, (2, 5), seed=0, image_dir=os.path.join(SMOKE_DIR, "Images"), size=(200, 150))
    print("SMOKE MODE: synthetic grey images; every number below is meaningless.")
else:
    ensure_fetal_planes()
    meta = read_metadata(DATA_ROOT)
print(meta.shape)
meta.head()

In [ ]:
# Load the split manifest written by notebook 01 (or rebuild it identically from the metadata)
if os.path.exists(SPLITS_CSV) and MODE != "smoke":
    splits = load_split_manifest(SPLITS_CSV, DATA_ROOT)
else:
    splits = make_splits(meta, val_fraction=0.15 if MODE == "smoke" else 0.1, seed=SEED)
    save_split_manifest(splits, SPLITS_CSV)
if MODE == "demo":  # stratified patient subset for speed
    keep = splits.groupby("split", group_keys=False).apply(lambda d: d[d.Patient_num.isin(pd.Series(d.Patient_num.unique()).sample(frac=0.25, random_state=SEED))])
    splits = keep.reset_index(drop=True)
print(splits.groupby(["split"]).agg(images=("Image_name", "size"), patients=("Patient_num", "nunique")))

## Architecture
`omari_us.models.DenseNet161Classifier` wraps torchvision's DenseNet-161 with an explicit, non-inplace final ReLU (`model.act`).
torchvision applies `F.relu(..., inplace=True)` inside `forward`, which interferes with activation and gradient hooks. Keys are unchanged, so
`DenseNet161_Weights.IMAGENET1K_V1` loads directly.

In [ ]:
from omari_us.models import build_model
model = build_model("densenet161", pretrained=(MODE != "smoke"))
n = sum(p.numel() for p in model.parameters())
print(f"init: {model.init_source} | parameters: {n:,} (expected 26,485,254 with a 6-class head)")
assert n == 26_485_254
x = torch.randn(1, 3, 224, 224)
shapes = {}
hooks = [getattr(model.features, b).register_forward_hook(lambda m, i, o, b=b: shapes.__setitem__(b, tuple(o.shape))) for b in ["denseblock1", "denseblock2", "denseblock3", "denseblock4"]]
with torch.no_grad(): model(x)
for h in hooks: h.remove()
shapes

## Train: frozen encoder (linear probe) and full fine-tuning
Both regimes use the same patient-disjoint splits, class-weighted cross-entropy with label smoothing, cosine schedule with warm-up,
and model selection by **validation macro-F1**. The test split is evaluated once with the selected checkpoint.
Hyper-parameters come from `configs/densenet161.yaml`; `demo` shortens epochs, `smoke` uses random weights and 2 batches.

In [ ]:
from omari_us.training import TrainConfig, train
from omari_us.utils import load_yaml
C = load_yaml(os.path.join(REPO, "configs", "densenet161.yaml"))
def make_cfg(regime, label_fraction=1.0, seed=0):
    r = dict(C["regimes"][regime])
    cfg = TrainConfig(model="densenet161", regime=regime, label_fraction=label_fraction, seed=seed, out_dir=RUNS,
                      loss=C["loss"], weight_scheme=C["weight_scheme"], label_smoothing=C["label_smoothing"], patience=C["patience"],
                      input_overrides=C.get("input", {}), augment=C.get("augment", {}), **r)
    if MODE == "demo":
        cfg.epochs = min(cfg.epochs, 4)
    if MODE == "smoke":
        cfg.epochs, cfg.batch_size, cfg.max_train_batches, cfg.max_eval_batches = 2, 4, 2, 2
        cfg.pretrained, cfg.num_workers, cfg.amp = False, 0, False
    return cfg

results = {}
for regime in ["frozen", "finetune"]:
    t0 = time.time()
    results[regime] = train(make_cfg(regime), splits)
    s = results[regime]
    print(f"{regime:9s} best epoch {s['best_epoch']} | val macro-F1 {s['best_val_macro_f1']:.3f} | test macro-F1 {s['test']['macro_f1']:.3f} | {time.time()-t0:.0f}s")

In [ ]:
# Learning curves (written to history.json in each run folder)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, regime in zip(axes, ["frozen", "finetune"]):
    h = json.load(open(os.path.join(RUNS, results[regime]["run"], "history.json")))
    ax.plot(h["epoch"], h["train_loss"], label="train loss"); ax.plot(h["epoch"], h["val_loss"], label="val loss")
    ax2 = ax.twinx(); ax2.plot(h["epoch"], h["val_macro_f1"], "g--", label="val macro-F1"); ax2.set_ylim(0, 1)
    ax.set_title(regime); ax.set_xlabel("epoch"); ax.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout(); plt.show()

In [ ]:
from omari_us.evaluation import classification_report
z = np.load(os.path.join(RUNS, results["finetune"]["run"], "test_predictions.npz"), allow_pickle=True)
rep = classification_report(z["labels"], z["probs"].argmax(1))
print(f"test accuracy {rep['accuracy']:.3f} | macro-F1 {rep['macro_f1']:.3f} | balanced accuracy {rep['balanced_accuracy']:.3f}")
pd.DataFrame(rep["per_class"], index=CLASS_SHORT).round(3)

### Optional: label-efficiency sweep
Repeats training on patient-level label fractions (1–100%) × seeds. Expensive in `full` mode (18 runs per regime);
notebook 05 collects the `summary.json` files into the website's learning-curve figure.

In [ ]:
RUN_LABEL_SWEEP = False  #@param {type:"boolean"}
SWEEP_REGIME = "frozen"  #@param ["frozen", "finetune"]
if RUN_LABEL_SWEEP:
    fracs = C["label_fractions"] if MODE == "full" else [0.05, 0.25, 1.0]
    seeds = C["seeds"] if MODE == "full" else [0]
    for f in fracs:
        for s in seeds:
            r = train(make_cfg(SWEEP_REGIME, f, s), splits, progress=False)
            print(r["run"], round(r["test"]["macro_f1"], 4))

## Feature maps from the fine-tuned model
Top channels by mean activation at each dense block, for one test image. The website shows the same view from exported PNGs.

In [ ]:
from omari_us.models import load_finetuned
from omari_us.datasets import input_spec, build_transforms
from PIL import Image
ft, meta_ft = load_finetuned("densenet161", os.path.join(RUNS, results["finetune"]["run"], "best.pt"))
tf = build_transforms(input_spec("densenet161"), False)
row = splits[splits.split == "test"].iloc[0]
x = tf(Image.open(row.path).convert("L")).unsqueeze(0)
acts = {}
hooks = [getattr(ft.features, b).register_forward_hook(lambda m, i, o, b=b: acts.__setitem__(b, torch.relu(o)[0])) for b in ["denseblock1", "denseblock2", "denseblock3", "denseblock4"]]
with torch.no_grad(): ft(x)
for h in hooks: h.remove()
fig, axes = plt.subplots(4, 8, figsize=(15, 8))
for i, (b, a) in enumerate(acts.items()):
    top = a.mean((1, 2)).argsort(descending=True)[:8]
    for j, c in enumerate(top):
        axes[i, j].imshow(a[c], cmap="inferno"); axes[i, j].axis("off"); axes[i, j].set_title(f"{b[-6:]} #{int(c)}", fontsize=7)
plt.tight_layout(); plt.show()

Next: notebook 05 compares models; `scripts/export_website_artifacts.py` exports feature maps, attributions and predictions with provenance.

## References
- Burgos-Artizzu X.P. et al. *Evaluation of deep convolutional neural networks for automatic classification of common maternal fetal ultrasound planes.* Sci. Rep. 10, 10200 (2020). Dataset: Zenodo 3904280, CC BY 4.0.
- Huang G. et al. *Densely connected convolutional networks.* CVPR 2017.
- Dosovitskiy A. et al. *An image is worth 16×16 words.* ICLR 2021; Steiner A. et al. *How to train your ViT?* TMLR 2022.
- He K. et al. *Masked autoencoders are scalable vision learners.* CVPR 2022.
- Megahed Y. et al. *USF-MAE: Ultrasound self-supervised foundation model with masked autoencoding.* Biomed. Signal Process. Control 122, 110313 (2026). https://github.com/Yusufii9/USF-MAE
- Zhou B. et al. CAM, CVPR 2016 · Selvaraju R.R. et al. Grad-CAM, ICCV 2017 · Chattopadhay A. et al. Grad-CAM++, WACV 2018 · Abnar & Zuidema, attention rollout, ACL 2020 · Chefer H. et al., CVPR 2021 · Adebayo J. et al., sanity checks, NeurIPS 2018 · Petsiuk V. et al., RISE / deletion–insertion, BMVC 2018 · Guo C. et al., calibration, ICML 2017.